In [ ]:
# %%
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random
import ast

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

In [ ]:
# %%
sensor_df = pd.read_parquet("sensor.parquet")
event_df = pd.read_parquet("event.parquet")

label_df = pd.read_csv("label.csv")  # event_id, track_id

In [ ]:
def safe_parse_array(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return np.array(x, dtype=np.float32)

    if isinstance(x, str):
        try:
            return np.array(ast.literal_eval(x), dtype=np.float32)
        except:
            x = x.replace("[", "").replace("]", "")
            return np.fromstring(x, sep=" ", dtype=np.float32)

    return np.zeros(0, dtype=np.float32)

In [ ]:
LAT_MEAN = sensor_df["latitude"].mean()
LON_MEAN = sensor_df["longitude"].mean()

LAT_STD = sensor_df["latitude"].std() + 1e-6
LON_STD = sensor_df["longitude"].std() + 1e-6

def normalize_xy(xy):
    xy[:, 0] = (xy[:, 0] - LAT_MEAN) / LAT_STD
    xy[:, 1] = (xy[:, 1] - LON_MEAN) / LON_STD
    return xy

In [ ]:
# %%
def pad_or_truncate(seq, L=10):
    if len(seq) >= L:
        return seq[:L]
    else:
        pad = np.repeat(seq[-1:], L - len(seq), axis=0)
        return np.vstack([seq, pad])

def build_subtracks(sensor_df, L=10):
    sub_tracks = []

    for tid, g in sensor_df.groupby("trackId"):
        g = g.sort_values("timestamp")
        xy = g[["latitude", "longitude"]].values

        meta = g.iloc[0]

        for w in range(5, 11):
            for i in range(0, len(xy) - w + 1, 2):
                seg = xy[i:i+w].copy()
                seg = normalize_xy(seg)
                seg = pad_or_truncate(seg, L)

                sub_tracks.append({
                    "track_id": tid,
                    "segment": seg,

                    # 🔥 label features
                    "trackName": str(meta.get("trackName", "UNK")),
                    "mmsi": str(meta.get("mmsi", "UNK")),
                    "modeS": str(meta.get("modeS", "UNK")),
                    "mode3A": str(meta.get("mode3A", "UNK")),
                })

    return sub_tracks

sub_tracks = build_subtracks(sensor_df)

In [ ]:
# %%
def encode_report(row, L=10):
    lons = safe_parse_array(row["coors.longitudes"])
    lats = safe_parse_array(row["coors.latitudes"])

    n = min(len(lons), len(lats))
    if n == 0:
        xy = np.zeros((L, 2))
    else:
        xy = np.stack([lats[:n], lons[:n]], axis=1)

    if len(xy) == 1:
        xy = np.repeat(xy, L, axis=0)

    if len(xy) < L:
        pad = np.repeat(xy[-1:], L - len(xy), axis=0)
        xy = np.vstack([xy, pad])

    xy = xy[:L]
    xy = normalize_xy(xy)

    return {
        "segment": xy,
        "obj": str(row.get("obj", "UNK")),
        "entityId": str(row.get("entityId", "UNK")),
        "basedName": str(row.get("coors.properties.basedName", "UNK"))
    }

In [ ]:
def build_vocab(values):
    uniq = list(set(values))
    return {v: i+1 for i, v in enumerate(uniq)}  # 0 = padding

trackName_vocab = build_vocab([s["trackName"] for s in sub_tracks])
mmsi_vocab = build_vocab([s["mmsi"] for s in sub_tracks])
modeS_vocab = build_vocab([s["modeS"] for s in sub_tracks])
mode3A_vocab = build_vocab([s["mode3A"] for s in sub_tracks])

report_data = [encode_report(r) for _, r in event_df.iterrows()]

obj_vocab = build_vocab([r["obj"] for r in report_data])
entity_vocab = build_vocab([r["entityId"] for r in report_data])
based_vocab = build_vocab([r["basedName"] for r in report_data])

In [ ]:
# %%
track_seqs = []
track_feats = []

for s in sub_tracks:
    track_seqs.append(s["segment"])

    track_feats.append([
        trackName_vocab.get(s["trackName"], 0),
        mmsi_vocab.get(s["mmsi"], 0),
        modeS_vocab.get(s["modeS"], 0),
        mode3A_vocab.get(s["mode3A"], 0),
    ])

report_seqs = []
report_feats = []

for r in report_data:
    report_seqs.append(r["segment"])

    report_feats.append([
        obj_vocab.get(r["obj"], 0),
        entity_vocab.get(r["entityId"], 0),
        based_vocab.get(r["basedName"], 0),
    ])

In [ ]:
# %%
event_to_idx = {eid: i for i, eid in enumerate(event_df["event_id"])}

track_to_idxs = {}
for i, s in enumerate(sub_tracks):
    track_to_idxs.setdefault(s["track_id"], []).append(i)

pairs = []

for _, row in label_df.iterrows():
    eid = row["event_id"]
    tid = row["track_id"]

    if eid not in event_to_idx or tid not in track_to_idxs:
        continue

    rid = event_to_idx[eid]

    for tidx in track_to_idxs[tid]:
        pairs.append((rid, tidx))

In [ ]:
# %%
class HybridDataset(Dataset):
    def __init__(self, report_seqs, track_seqs, report_feats, track_feats, pairs):
        self.data = []

        for rid, tid in pairs:
            r = report_seqs[rid]
            p = track_seqs[tid]

            rf = report_feats[rid]
            pf = track_feats[tid]

            for _ in range(3):
                neg = random.randint(0, len(track_seqs)-1)

                if neg == tid:
                    continue

                self.data.append((r, p, track_seqs[neg], rf, pf, track_feats[neg]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        r, p, n, rf, pf, nf = self.data[idx]

        return (
            torch.tensor(r, dtype=torch.float32),
            torch.tensor(p, dtype=torch.float32),
            torch.tensor(n, dtype=torch.float32),
            torch.tensor(rf),
            torch.tensor(pf),
            torch.tensor(nf),
        )

In [ ]:
# %%
class Model(nn.Module):
    def __init__(self, d_model=64):
        super().__init__()

        self.input_proj = nn.Linear(2, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # 🔥 feature embeddings
        self.feat_emb = nn.Embedding(10000, d_model)

        # adaptor
        self.adaptor = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )

    def encode_traj(self, x):
        x = self.input_proj(x)
        x = self.encoder(x)
        return x.mean(dim=1)

    def encode_feat(self, f):
        emb = self.feat_emb(f)  # (B, F, D)
        return emb.mean(dim=1)

    def forward(self, x, feat):
        traj = self.encode_traj(x)
        feat = self.encode_feat(feat)

        combined = torch.cat([traj, 0.3 * feat], dim=1)
        sem = self.adaptor(combined)

        return traj, sem

In [ ]:
# %%
net = Model()

dataset = HybridDataset(report_seqs, track_seqs, report_feats, track_feats, pairs)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

opt = torch.optim.Adam(net.parameters(), lr=1e-3)

for ep in range(20):
    total = 0

    for r, p, n, rf, pf, nf in loader:
        opt.zero_grad()

        tr_r, sem_r = net(r, rf)
        tr_p, sem_p = net(p, pf)
        tr_n, sem_n = net(n, nf)

        # metric loss
        e_r = F.normalize(tr_r, dim=1)
        e_p = F.normalize(tr_p, dim=1)
        e_n = F.normalize(tr_n, dim=1)

        loss_metric = F.relu(
            F.cosine_similarity(e_r, e_n) -
            F.cosine_similarity(e_r, e_p) + 0.2
        ).mean()

        # semantic loss
        loss_sem = (1 - F.cosine_similarity(
            F.normalize(sem_r, dim=1),
            F.normalize(sem_p, dim=1)
        )).mean()

        loss = loss_metric + 0.5 * loss_sem

        loss.backward()
        opt.step()

        total += loss.item()

    print(ep, total/len(loader))